In [1]:
# -*- coding: utf-8 -*-
"""
[별도 스크립트] Test 최종 평가표 (HTML 자동 오픈)
- 기존 체크포인트로 Test 성능 계산
- 표 1: 발전소별 성능(Test) + 가중 평균
- 표 2: 지역별 성능(Test) + 가중 평균
- 주피터 표 표시 + HTML/CSV 저장 + 실행 후 자동 웹브라우저 오픈
"""

import os, warnings, webbrowser
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from jinja2 import Template

warnings.filterwarnings("ignore")

# =========================
# 경로와 설정
# =========================
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
VAL_CSV   = r"C:\ESG_Project1\file\merge_data\val.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"

CKPT_ROOT = r"C:\ESG_Project1\xgboost\output\1w_feat22_rep\checkpoints"
SAVE_DIR  = os.path.join(os.path.dirname(CKPT_ROOT), "Eval_HTML")
os.makedirs(SAVE_DIR, exist_ok=True)

CONTEXT_2W   = False
FEATURE_MODE = 22
USE_DELTAS   = True

USE_HOURLY_CALIB = True
CALIB_CLIP = 0.25

TIME_COL, GROUP_COL, REGION_COL = "일시", "발전구분", "지역"
TARGET_HOURLY = "합산발전량(MWh)"
TARGET_NEXT24 = "target_next_24h"

WEATHER_COLS = ["기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
                "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

# =========================
# 주피터 표시 유틸
# =========================
JUPYTER_MODE = True
try:
    from IPython.display import display, HTML
except Exception:
    JUPYTER_MODE = False
    def display(*args, **kwargs): pass
    def HTML(x): return x

def show_df(df: pd.DataFrame, title: str = None):
    if title:
        if JUPYTER_MODE: display(HTML(f"<h3>{title}</h3>"))
        else: print(f"\n[{title}]")
    if JUPYTER_MODE: display(df)
    else:
        try: print(df.to_string(index=False))
        except: print(df.head())

# =========================
# 피처 구성
# =========================
def _lags_for_48():
    return [1,2,3,6,12,18,24,36,48,72,96,120,144,168,240,288,336]

def _resolve_context_lags(context_2w: bool):
    return _lags_for_48() if context_2w else [1,3,6,24]

if FEATURE_MODE == 22:
    LAGS_FEATURE = [1,3,6,24]
    USE_PAST_DELTAS = True if USE_DELTAS else False
elif FEATURE_MODE == 48:
    LAGS_FEATURE = _lags_for_48()
    USE_PAST_DELTAS = True
else:
    raise ValueError("FEATURE_MODE must be 22 or 48")

LAGS_CONTEXT = _resolve_context_lags(CONTEXT_2W)
LAGS_BUILD   = sorted(set(LAGS_CONTEXT).union(LAGS_FEATURE))

def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"]  = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"]  = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"]  = np.cos(2*np.pi*df["doy"]/365)
    return df

def add_lags(df, lags):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    ext = sorted(set(lags + [k+1 for k in lags]))
    for k in ext:
        df[f"lag_{k}"] = df.groupby(GROUP_COL)[TARGET_HOURLY].shift(k)
    return df

def add_deltas(df, lags, use=True):
    if not use: return df
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    for k in lags:
        a, b = "lag_1", f"lag_{k+1}"
        if a in df.columns and b in df.columns:
            df[f"diff_past_{k}"] = df[a] - df[b]
    return df

def make_next24_target(df):
    df = df.copy()
    g = df.groupby(GROUP_COL)[TARGET_HOURLY]
    df[TARGET_NEXT24] = (g.shift(-1).rolling(24, min_periods=24).sum()
                         .reset_index(level=0, drop=True))
    return df

def build_table(df):
    df = add_time_feats(df)
    df = add_lags(df, LAGS_BUILD)
    df = add_deltas(df, LAGS_BUILD, USE_PAST_DELTAS)
    df = make_next24_target(df)
    df.dropna(subset=[TARGET_NEXT24], inplace=True)
    df.fillna(0.0, inplace=True)
    return df

def dmat(X): return xgb.DMatrix(X)

# =========================
# 데이터 로드
# =========================
train_raw = pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])
val_raw   = pd.read_csv(VAL_CSV,   parse_dates=[TIME_COL])
test_raw  = pd.read_csv(TEST_CSV,  parse_dates=[TIME_COL])

train_raw = build_table(train_raw)
val_raw   = build_table(val_raw)
test_raw  = build_table(test_raw)

lag_cols  = [f"lag_{k}" for k in LAGS_FEATURE]
diff_cols = [f"diff_past_{k}" for k in LAGS_FEATURE] if USE_PAST_DELTAS else []
all_candidates = WEATHER_COLS + TIME_FEATS + lag_cols + diff_cols
feature_cols = [c for c in all_candidates if c in train_raw.columns]

# =========================
# 스케일러 구성
# =========================
region_scalers = {}
for region, grp in train_raw.groupby(REGION_COL):
    X = grp[feature_cols].to_numpy(np.float32)
    y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
    std = StandardScaler().fit(X)
    mm  = MinMaxScaler().fit(std.transform(X))
    tsc = StandardScaler().fit(y)
    region_scalers[region] = (std, mm, tsc)

def apply_transform(df):
    df = df.copy()
    for region, grp in df.groupby(REGION_COL):
        if region not in region_scalers: continue
        std, mm, tsc = region_scalers[region]
        X = grp[feature_cols].to_numpy(np.float32)
        y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
        df.loc[grp.index, feature_cols]  = mm.transform(std.transform(X))
        df.loc[grp.index, TARGET_NEXT24] = tsc.transform(y)
    return df

train = apply_transform(train_raw)
val   = apply_transform(val_raw)
test  = apply_transform(test_raw)

# =========================
# 메트릭과 보정
# =========================
def inverse_target(region, y_scaled):
    _, _, tsc = region_scalers[region]
    y_log = tsc.inverse_transform(np.asarray(y_scaled).reshape(-1,1)).ravel()
    y = np.expm1(y_log) - 1e-8
    return np.clip(y, 0, None)

def metrics_mwh(region, y_true_scaled, y_pred_scaled, return_series=False):
    y_true = inverse_target(region, y_true_scaled)
    y_pred = inverse_target(region, y_pred_scaled)
    r2  = max(r2_score(y_true, y_pred), 0.0)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    if return_series: return r2, rmse, mae, y_true, y_pred
    return r2, rmse, mae

def _file_ok(p):
    try: return os.path.exists(p) and os.path.getsize(p) > 0
    except: return False

def load_booster(p):
    if not _file_ok(p): return None
    try:
        b = xgb.Booster(); b.load_model(p); return b
    except: return None

def plant_model_path(plant):
    p = os.path.join(CKPT_ROOT, "plant", plant, "best.json")
    return p if _file_ok(p) else None

def region_model_path(region):
    p = os.path.join(CKPT_ROOT, "region", region, "best.json")
    return p if _file_ok(p) else None

def compute_hour_bias(region, booster, df_val_region):
    if df_val_region.empty or booster is None: return {}
    Xv = df_val_region[feature_cols].to_numpy(np.float32)
    yv = df_val_region[TARGET_NEXT24].to_numpy(np.float32)
    y_pred_scaled = booster.predict(dmat(Xv))
    _, _, _, y_true, y_pred = metrics_mwh(region, yv, y_pred_scaled, return_series=True)
    hours = df_val_region["hour"].to_numpy()
    tmp = pd.DataFrame({"hour": hours, "res": y_true - y_pred})
    return tmp.groupby("hour")["res"].mean().to_dict()

def apply_hour_bias(region, y_pred, hours, bias):
    if not USE_HOURLY_CALIB or not bias: return y_pred
    out = y_pred.copy()
    for i, h in enumerate(hours):
        b = bias.get(int(h), 0.0)
        cap = max(1e-8, abs(out[i]) * CALIB_CLIP)
        out[i] = max(0.0, out[i] + max(-cap, min(cap, b)))
    return out

def _weighted_avg(vals: np.ndarray, w: np.ndarray) -> float:
    wsum = float(w.sum()) if float(w.sum()) > 0 else 1.0
    return float((vals * w).sum() / wsum)

# =========================
# Validation에서 지역별 시간대 보정치 계산
# =========================
region_bias = {}
for region, df_va_r in val.groupby(REGION_COL):
    bias = compute_hour_bias(region, load_booster(region_model_path(region)), df_va_r)
    region_bias[region] = bias

# =========================
# 1) 발전소별 성능(Test)
# =========================
plant_rows_test_adj = []
for plant, df_te in test.groupby(GROUP_COL):
    ppath = plant_model_path(plant)
    if not ppath: continue
    booster = load_booster(ppath)
    if booster is None or len(df_te) == 0: continue
    region = df_te[REGION_COL].iloc[0]
    Xte = df_te[feature_cols].to_numpy(np.float32)
    yte = df_te[TARGET_NEXT24].to_numpy(np.float32)
    y_pred_scaled = booster.predict(dmat(Xte))
    _, _, _, y_true_mwh, y_pred_mwh = metrics_mwh(region, yte, y_pred_scaled, return_series=True)
    hours = df_te["hour"].to_numpy()
    bias = region_bias.get(region, {})
    y_pred_adj = apply_hour_bias(region, y_pred_mwh, hours, bias)
    r2 = max(r2_score(y_true_mwh, y_pred_adj), 0.0)
    rmse = float(np.sqrt(mean_squared_error(y_true_mwh, y_pred_adj)))
    mae  = float(mean_absolute_error(y_true_mwh, y_pred_adj))
    plant_rows_test_adj.append({"plant": plant, "region": region, "test_r2": r2, "test_rmse": rmse, "test_mae": mae})

df_plant_test = pd.DataFrame(plant_rows_test_adj)
if not df_plant_test.empty:
    df_plant_test = df_plant_test.sort_values(["region","test_r2"], ascending=[True, False]).reset_index(drop=True)
    w_plant = np.array([len(test[test[GROUP_COL] == p]) for p in df_plant_test["plant"]], dtype=float)
    wr2_plant   = _weighted_avg(df_plant_test["test_r2"].to_numpy(float),   w_plant)
    wrmse_plant = _weighted_avg(df_plant_test["test_rmse"].to_numpy(float), w_plant)
    wmae_plant  = _weighted_avg(df_plant_test["test_mae"].to_numpy(float),  w_plant)
else:
    wr2_plant = wrmse_plant = wmae_plant = float('nan')

plant_table = pd.DataFrame()
if not df_plant_test.empty:
    plant_table = (
        df_plant_test[["plant","region","test_r2","test_rmse","test_mae"]]
        .rename(columns={"plant":"발전소","region":"지역","test_r2":"R²","test_rmse":"RMSE","test_mae":"MAE"})
        .sort_values(["지역","R²"], ascending=[True, False])
        .reset_index(drop=True)
    )
    plant_table = pd.concat([
        plant_table,
        pd.DataFrame([{"발전소":"가중 평균","지역":"","R²":wr2_plant,"RMSE":wrmse_plant,"MAE":wmae_plant}])
    ], ignore_index=True)

# =========================
# 2) 지역별 성능(Test)
# =========================
region_rows_test_post = []
for region, df_te_r in test.groupby(REGION_COL):
    booster_r = load_booster(region_model_path(region))
    if booster_r is None or len(df_te_r) == 0: continue
    Xte = df_te_r[feature_cols].to_numpy(np.float32)
    yte = df_te_r[TARGET_NEXT24].to_numpy(np.float32)
    y_pred_scaled = booster_r.predict(dmat(Xte))
    _, _, _, y_true, y_pred = metrics_mwh(region, yte, y_pred_scaled, return_series=True)
    hours = df_te_r["hour"].to_numpy()
    bias = region_bias.get(region, {})
    y_pred_adj = apply_hour_bias(region, y_pred, hours, bias)
    r2 = max(r2_score(y_true, y_pred_adj), 0.0)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred_adj)))
    mae  = float(mean_absolute_error(y_true, y_pred_adj))
    region_rows_test_post.append({"region": region, "r2": r2, "rmse": rmse, "mae": mae})

df_region_test = pd.DataFrame(region_rows_test_post).sort_values("r2", ascending=False).reset_index(drop=True)

if not df_region_test.empty:
    w_region = np.array([len(test[test[REGION_COL] == r]) for r in df_region_test["region"]], dtype=float)
    wr2_region   = _weighted_avg(df_region_test["r2"].to_numpy(float),  w_region)
    wrmse_region = _weighted_avg(df_region_test["rmse"].to_numpy(float), w_region)
    wmae_region  = _weighted_avg(df_region_test["mae"].to_numpy(float),  w_region)
else:
    wr2_region = wrmse_region = wmae_region = float('nan')

region_table = pd.DataFrame()
if not df_region_test.empty:
    region_table = (
        df_region_test[["region","r2","rmse","mae"]]
        .rename(columns={"region":"지역","r2":"R²","rmse":"RMSE","mae":"MAE"})
        .sort_values("R²", ascending=False)
        .reset_index(drop=True)
    )
    region_table = pd.concat([
        region_table,
        pd.DataFrame([{"지역":"가중 평균","R²":wr2_region,"RMSE":wrmse_region,"MAE":wmae_region}])
    ], ignore_index=True)

# =========================
# 주피터 표시 + CSV 저장
# =========================
if not plant_table.empty:
    show_df(plant_table, "표 1) 발전소별 성능 평가")
    try: plant_table.to_csv(os.path.join(SAVE_DIR, "table_test_plants.csv"), index=False, encoding="utf-8-sig")
    except Exception: pass
else:
    print("⚠ 발전소별 Test 표를 만들 데이터가 없습니다.")

if not region_table.empty:
    show_df(region_table, "표 2) 지역별 성능 평가")
    try: region_table.to_csv(os.path.join(SAVE_DIR, "table_test_regions.csv"), index=False, encoding="utf-8-sig")
    except Exception: pass
else:
    print("⚠ 지역별 Test 표를 만들 데이터가 없습니다.")

# =========================
# HTML 저장 + 자동 오픈
# =========================
def fmt3(v):
    try: return f"{v:.3f}"
    except: return "-"

def write_html(path):
    from jinja2 import Template
    tpl = Template(r"""
<html lang="ko"><head><meta charset="utf-8">
<title>발전소 및 지역별 성능 평가</title>
<style>
body{font-family:Arial, sans-serif; margin:18px;}
table{border-collapse:collapse; width:100%; margin:16px 0;}
th,td{border:1px solid #bbb; padding:6px 8px; text-align:center;}
th{background:#7FFF00;}
h1{margin-top:8px;}
h2{margin-top:28px;}
.bold-row td{font-weight:bold; background:#f9f9f9;}
</style></head>
<body>
<h1 style="text-align:center;">🌟 발전소 및 지역별 성능 평가 🌟</h1>

{% if plant_rows|length > 0 %}
<h2>1️⃣ 발전소별 성능</h2>
<table>
<tr><th>발전소</th><th>지역</th><th>R²</th><th>RMSE</th><th>MAE</th></tr>
{% for r in plant_rows %}
<tr {% if r.is_weighted %}class="bold-row"{% endif %}>
<td>{{r.plant}}</td><td>{{r.region}}</td><td>{{r.r2}}</td><td>{{r.rmse}}</td><td>{{r.mae}}</td>
</tr>
{% endfor %}
</table>
{% endif %}

{% if region_rows|length > 0 %}
<h2>2️⃣ 지역별 성능</h2>
<table>
<tr><th>지역</th><th>R²</th><th>RMSE</th><th>MAE</th></tr>
{% for r in region_rows %}
<tr {% if r.is_weighted %}class="bold-row"{% endif %}>
<td>{{r.region}}</td><td>{{r.r2}}</td><td>{{r.rmse}}</td><td>{{r.mae}}</td>
</tr>
{% endfor %}
</table>
{% endif %}
</body></html>
""")

    # plant rows
    plant_rows = []
    if not plant_table.empty:
        for _, r in plant_table.iterrows():
            plant_rows.append(dict(
                plant=str(r["발전소"]),
                region=str(r["지역"]),
                r2=f"{r['R²']:.3f}" if pd.notna(r["R²"]) else "-",
                rmse=f"{r['RMSE']:.3f}" if pd.notna(r["RMSE"]) else "-",
                mae=f"{r['MAE']:.3f}" if pd.notna(r["MAE"]) else "-",
                is_weighted=(r["발전소"] == "가중 평균")
            ))

    # region rows
    region_rows = []
    if not region_table.empty:
        for _, r in region_table.iterrows():
            region_rows.append(dict(
                region=str(r["지역"]),
                r2=f"{r['R²']:.3f}" if pd.notna(r["R²"]) else "-",
                rmse=f"{r['RMSE']:.3f}" if pd.notna(r["RMSE"]) else "-",
                mae=f"{r['MAE']:.3f}" if pd.notna(r["MAE"]) else "-",
                is_weighted=(r["지역"] == "가중 평균")
            ))

    html = tpl.render(plant_rows=plant_rows, region_rows=region_rows)
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ HTML 저장: {path}")

    # 자동 오픈
    try:
        import webbrowser, os
        webbrowser.open("file://" + os.path.abspath(path))
    except Exception:
        pass



html_path = os.path.join(SAVE_DIR, "test_tables.html")
write_html(html_path)

print("\n🏁 완료")


,발전소,지역,R²,RMSE,MAE
0,남제주소내,고산,0.737340,0.144308,0.102084
1,하동본부,광양시,0.658188,5.095719,3.636212
2,하동정수장,광양시,0.478676,0.099736,0.076128
3,하동보건소,광양시,0.370765,0.048631,0.038024
4,하동하수처리장,광양시,0.216806,0.071873,0.059157
5,하동공설운동장,광양시,0.130463,0.379766,0.297170
6,하동변전소,광양시,0.025490,0.067271,0.056713
7,부산운동장,부산,0.729811,1.137304,0.772884
8,부산수처리장,부산,0.642088,0.102498,0.076201
9,부산본부,부산,0.639240,0.519956,0.372711


,지역,R²,RMSE,MAE
0,광양시,0.763404,3.950159,1.225775
1,고산,0.737377,0.144298,0.102063
2,부산,0.736187,1.145892,0.544230
3,울진,0.702884,5.965410,4.391958
4,성산,0.699557,0.994358,0.558190
5,영월,0.682303,1.066207,0.780268
6,인천,0.452481,1.608368,1.033999
7,가중 평균,0.691604,2.285271,1.029919


✅ HTML 저장: C:\ESG_Project1\xgboost\output\1w_feat22_rep\Eval_HTML\test_tables.html

🏁 완료


In [ ]:
# -*- coding: utf-8 -*-
"""
XGBoost 지역 모델 기반 이상치 탐지 (Test 기준)
- ckpt 경로의 지역별 best.json 로드
- 이상치: '행(발전소×시간)' 절대오차 상위 TOP_P 퍼센트
- 표: 지역별 (표본수=행수, 이상치 개수/비율, Top3 영향요인) + '합계'
- 그래프: 전체(합산) + 지역별 (X: Time, Y: 합산 발전량(MWh), Outliers=빨간점, 행 기준)
- 주피터 표시는 display, HTML 리포트 자동 오픈
"""

import os, warnings, webbrowser
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from jinja2 import Template
from matplotlib.lines import Line2D   # ★ 커스텀 범례용 추가

warnings.filterwarnings("ignore")

# =========================
# 경로 / 설정
# =========================
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"
CKPT_ROOT = r"C:\ESG_Project1\xgboost\output\1w_feat22_rep\checkpoints"

REPORT_DIR = os.path.join(os.path.dirname(CKPT_ROOT), "Outliers_HTML")
PLOTS_DIR  = os.path.join(REPORT_DIR, "plots")
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

TOP_P = 0.03  # 상위 3%
LINE_W = 0.9  # ★ 선 굵기(얇게)
DOT_SIZE = 6  # ★ 이상치 점 크기(작게)
TIME_COL, GROUP_COL, REGION_COL = "일시", "발전구분", "지역"
TARGET_HOURLY = "합산발전량(MWh)"
TARGET_NEXT24 = "target_next_24h"

WEATHER_COLS = ["기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
                "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

CONTEXT_2W = False
FEATURE_MODE = 22  # {22, 48}
USE_PAST_DELTAS = True

def _lags_for_48(): return [1,2,3,6,12,18,24,36,48,72,96,120,144,168,240,288,336]
if FEATURE_MODE == 22:
    LAGS_FEATURE = [1,3,6,24]
elif FEATURE_MODE == 48:
    LAGS_FEATURE = _lags_for_48(); USE_PAST_DELTAS = True
else:
    raise ValueError("FEATURE_MODE must be 22 or 48")

LAGS_CONTEXT = [1,3,6,24] if not CONTEXT_2W else _lags_for_48()
LAGS_BUILD = sorted(set(LAGS_CONTEXT).union(LAGS_FEATURE))

# 주피터 표시 헬퍼
JUPYTER_MODE = True
try:
    from IPython.display import display, HTML
except Exception:
    JUPYTER_MODE = False
    def display(*args, **kwargs): pass
    def HTML(x): return x

def show_df(df: pd.DataFrame, title: str = None, head=50):
    if title:
        if JUPYTER_MODE: display(HTML(f"<h3>{title}</h3>"))
        else: print(f"\n[{title}]")
    if JUPYTER_MODE: display(df.head(head))
    else:
        try: print(df.head(head).to_string(index=False))
        except: print(df.head(head))

# 폰트(한글)
def _set_korean_font():
    try: matplotlib.rc("font", family="Malgun Gothic")
    except Exception:
        try: matplotlib.rc("font", family="AppleGothic")
        except Exception: matplotlib.rc("font", family="DejaVu Sans")
    matplotlib.rcParams["axes.unicode_minus"] = False
_set_korean_font()

# =========================
# 피처/타깃 구성
# =========================
def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"]  = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"]  = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"]  = np.cos(2*np.pi*df["doy"]/365)
    return df

def add_lags(df, lags):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    ext = sorted(set(lags + [k+1 for k in lags]))
    for k in ext:
        df[f"lag_{k}"] = df.groupby(GROUP_COL)[TARGET_HOURLY].shift(k)
    return df

def add_deltas(df, lags, use=True):
    if not use: return df
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    for k in lags:
        a, b = "lag_1", f"lag_{k+1}"
        if a in df.columns and b in df.columns:
            df[f"diff_past_{k}"] = df[a] - df[b]
    return df

def make_next24_target(df):
    df = df.copy()
    g = df.groupby(GROUP_COL)[TARGET_HOURLY]
    df[TARGET_NEXT24] = (g.shift(-1).rolling(24, min_periods=24).sum()
                         .reset_index(level=0, drop=True))
    return df

def build_table(df):
    df = add_time_feats(df)
    df = add_lags(df, LAGS_BUILD)
    df = add_deltas(df, LAGS_BUILD, USE_PAST_DELTAS)
    df = make_next24_target(df)
    df.dropna(subset=[TARGET_NEXT24], inplace=True)
    df.fillna(0.0, inplace=True)
    return df

# =========================
# 데이터 로드 & 스케일
# =========================
train_raw = pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])
test_raw  = pd.read_csv(TEST_CSV,  parse_dates=[TIME_COL])

train_raw = build_table(train_raw)
test_raw  = build_table(test_raw)

lag_cols  = [f"lag_{k}" for k in LAGS_FEATURE]
diff_cols = [f"diff_past_{k}" for k in LAGS_FEATURE] if USE_PAST_DELTAS else []
all_candidates = WEATHER_COLS + TIME_FEATS + lag_cols + diff_cols
feature_cols = [c for c in all_candidates if c in train_raw.columns]

# 지역별 스케일러
region_scalers: dict[str, tuple[StandardScaler, MinMaxScaler, StandardScaler]] = {}
for region, grp in train_raw.groupby(REGION_COL):
    X = grp[feature_cols].to_numpy(np.float32)
    y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
    std = StandardScaler().fit(X)
    mm  = MinMaxScaler().fit(std.transform(X))
    tsc = StandardScaler().fit(y)
    region_scalers[region] = (std, mm, tsc)

def transform_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for region, grp in df.groupby(REGION_COL):
        if region not in region_scalers: continue
        std, mm, tsc = region_scalers[region]
        X = grp[feature_cols].to_numpy(np.float32)
        y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
        df.loc[grp.index, feature_cols]  = mm.transform(std.transform(X))
        df.loc[grp.index, TARGET_NEXT24] = tsc.transform(y)
    return df

test = transform_df(test_raw)

def inverse_target(region: str, y_scaled):
    _, _, tsc = region_scalers[region]
    y_log = tsc.inverse_transform(np.asarray(y_scaled).reshape(-1,1)).ravel()
    y = np.expm1(y_log) - 1e-8
    return np.clip(y, 0, None)

def dmat(X): return xgb.DMatrix(X)

def region_model_path(region: str):
    p = os.path.join(CKPT_ROOT, "region", region, "best.json")
    return p if os.path.exists(p) and os.path.getsize(p) > 0 else None

def load_booster(p):
    try: b = xgb.Booster(); b.load_model(p); return b
    except Exception: return None

# =========================
# 이상치 탐지 (행 기준) + 그래프(행 기준으로 점 표시)
# =========================
summary_rows = []
region_plot_paths = {}
overall_png = None

# 전체 그래프용 누적(시간 합산) + 행 기준 이상치 시각 리스트
overall_true = {}
overall_pred = {}
overall_outlier_times_rows = []   # ★ 행 기준 전체 이상치 시각(중복 허용)

for region, df_te_r in test.groupby(REGION_COL):
    rpath = region_model_path(region)
    booster_r = load_booster(rpath) if rpath else None
    if booster_r is None or len(df_te_r) == 0:
        continue

    # --- 예측 (행 기준) ---
    Xte = df_te_r[feature_cols].to_numpy(np.float32)
    yte = df_te_r[TARGET_NEXT24].to_numpy(np.float32)
    y_pred_scaled = booster_r.predict(dmat(Xte))

    y_true_row = inverse_target(region, yte)              # 각 행(발전소×시간)
    y_pred_row = inverse_target(region, y_pred_scaled)    # 각 행(발전소×시간)
    ae_row = np.abs(y_true_row - y_pred_row)

    # --- 지역 임계값 & 이상치 (행 기준) ---
    thr = float(np.quantile(ae_row, 1.0 - TOP_P)) if len(ae_row) else 0.0
    row_mask = ae_row >= thr
    n_out = int(row_mask.sum())
    n_rows = int(len(ae_row))
    ratio = 100.0 * n_out / max(1, n_rows)

    # --- Top3 영향요인(간단: |corr| 상위) ---
    try:
        X_df = pd.DataFrame(Xte, columns=feature_cols)
        corr = X_df.apply(lambda s: np.corrcoef(s.values, y_pred_row)[0,1])
        top3 = ", ".join(corr.abs().sort_values(ascending=False).index[:3].tolist())
    except Exception:
        top3 = "-"

    # --- 그래프용: 시간별 합산 + 이상치(행 기준) 시각들 ---
    df_tmp = pd.DataFrame({
        "t": df_te_r[TIME_COL].to_numpy(),
        "y_true": y_true_row,
        "y_pred": y_pred_row,
        "is_out": row_mask
    })
    # 시간별 합산
    s_true = df_tmp.groupby("t", as_index=True)["y_true"].sum().sort_index()
    s_pred = df_tmp.groupby("t", as_index=True)["y_pred"].sum().sort_index()

    # 지역 그래프(행 기준으로 점 모두 표시)
    out_times_rows = df_tmp.loc[df_tmp["is_out"], "t"].tolist()  # 중복 허용
    y_vals = [s_true.get(pd.to_datetime(t), np.nan) for t in out_times_rows]
    y_vals = np.array(y_vals, dtype=float)
    valid_mask = ~np.isnan(y_vals)
    x_points = pd.to_datetime(out_times_rows).to_numpy()[valid_mask]
    y_points = y_vals[valid_mask]

    fig = plt.figure(figsize=(10.5, 4.2))
    ax = plt.gca()
    ax.plot(s_true.index, s_true.values, linewidth=LINE_W, label="Actual")
    ax.plot(s_pred.index, s_pred.values, linewidth=LINE_W, label="Predicted")
    if len(x_points) > 0:
        ax.scatter(x_points, y_points, s=DOT_SIZE, alpha=0.35, marker="o", c="red",
                   label=f"Outliers ({len(x_points)})")
    ax.set_title(f"[{region}] Actual vs Predicted (이상치 표시)")
    ax.set_xlabel("Time"); ax.set_ylabel("합산 발전량(MWh)")
    ax.grid(True, alpha=0.25)

   
    outlier_label = f"Outliers ({len(x_points)})"  # ★ 개수 라벨 복원

    # ★ 커스텀 범례(빨간 점 + 개수)
    legend_elements = [
        Line2D([0], [0], color='C0', lw=LINE_W, label='Actual'),
        Line2D([0], [0], color='C1', lw=LINE_W, label='Predicted'),
        Line2D([0], [0], marker='o', color='red', lw=0, markersize=5, label=outlier_label),
    ]
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1,1), framealpha=0.9)

    fig.tight_layout()
    out_path = os.path.join(PLOTS_DIR, f"{region}_outlier_line.png")
    fig.savefig(out_path, dpi=130); plt.close(fig)
    region_plot_paths[region] = out_path

    # 요약 행 (표본수=행 수, 이상치=행 기준)
    summary_rows.append({
        "지역": region,
        "표본수": n_rows,
        "이상치 개수(비율)": f"{n_out} ({ratio:.3f}%)",
        "Top3 영향 요인": top3
    })

    # 전체 그래프 누적
    for t, v in s_true.items():
        overall_true[t] = overall_true.get(t, 0.0) + float(v)
    for t, v in s_pred.items():
        overall_pred[t] = overall_pred.get(t, 0.0) + float(v)
    overall_outlier_times_rows.extend(out_times_rows)  # 행 기준 시각 전부 누적

# =========================
# 전체(합산) 그래프 (행 기준으로 점 모두 표시)
# =========================
if len(overall_true) > 0:
    s_true_all = pd.Series(overall_true).sort_index()
    s_pred_all = pd.Series(overall_pred).sort_index()

    out_t_all_rows = pd.to_datetime(overall_outlier_times_rows)
    y_vals_all = [s_true_all.get(t, np.nan) for t in out_t_all_rows]
    y_vals_all = np.array(y_vals_all, dtype=float)
    valid_mask = ~np.isnan(y_vals_all)
    x_points_all = out_t_all_rows.to_numpy()[valid_mask]
    y_points_all = y_vals_all[valid_mask]

    fig = plt.figure(figsize=(11.5, 4.4))
    ax = plt.gca()
    ax.plot(s_true_all.index, s_true_all.values, linewidth=LINE_W, label="Actual")
    ax.plot(s_pred_all.index, s_pred_all.values, linewidth=LINE_W, label="Predicted")
    if len(x_points_all) > 0:
        ax.scatter(x_points_all, y_points_all, s=DOT_SIZE, alpha=0.35, marker="o", c="red",
                   label=f"Outliers {len(x_points_all)})")
    ax.set_title("합계 Actual vs Predicted (이상치 표시)")
    ax.set_xlabel("Time"); ax.set_ylabel("합산 발전량(MWh)")
    ax.grid(True, alpha=0.25)

    # ★ 커스텀 범례(빨간 점 고정)
    outlier_label_all = f"Outliers ({len(x_points_all)})"  # ★ 개수 라벨 복원

    legend_elements = [
        Line2D([0], [0], color='C0', lw=LINE_W, label='Actual'),
        Line2D([0], [0], color='C1', lw=LINE_W, label='Predicted'),
        Line2D([0], [0], marker='o', color='red', lw=0, markersize=5, label=outlier_label_all),
    ]
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1,1), framealpha=0.9)

    fig.tight_layout()
    overall_png = os.path.join(PLOTS_DIR, "overall_outlier_line.png")
    fig.savefig(overall_png, dpi=130); plt.close(fig)

# =========================
# 표 생성 (지역별 + 합계)
# =========================
df_summary = pd.DataFrame(summary_rows).sort_values("지역").reset_index(drop=True)
if not df_summary.empty:
    total_n = int(df_summary["표본수"].sum())
    total_k = int(df_summary["이상치 개수(비율)"].str.extract(r"(\d+)")[0].astype(int).sum())
    total_ratio = 100.0 * total_k / max(1, total_n)
    sum_row = pd.DataFrame([{
        "지역": "합계",
        "표본수": total_n,
        "이상치 개수(비율)": f"{total_k} ({total_ratio:.3f}%)",
        "Top3 영향 요인": "-"
    }])
    df_summary = pd.concat([df_summary, sum_row], ignore_index=True)
    show_df(df_summary, "요약 표 (지역별 + 합계)", head=100)

# =========================
# HTML 리포트
# =========================
HTML_PATH = os.path.join(REPORT_DIR, "outliers_report.html")

tpl = Template(r"""
<html lang="ko"><head><meta charset="utf-8">
<title>이상치 탐지 리포트</title>
<style>
body{font-family:Arial, sans-serif; margin:18px;}
h1{margin:0 0 6px 0;}
h2{margin-top:28px;}
table{border-collapse:collapse; width:100%; margin:12px 0;}
th,td{border:1px solid #bbb; padding:6px 8px; text-align:center;}
th{background:#7FFF00;}
tr.sum-row td{font-weight:bold;}
img{max-width:100%; height:auto; border:1px solid #e3e3e3; border-radius:6px;}
.small{color:#666; font-size:12px; margin:6px 0 14px 0;}
</style></head>
<body>
<h1 style="text-align:center;">🌟 발전소 및 지역별 이상치 탐지 결과 🌟</h1>

{% if overall_img %}
<h2>1️⃣ 합계 그래프</h2>
<img src="{{overall_img}}" alt="overall">
{% endif %}

{% if rows|length > 0 %}
<h2>2️⃣ 요약 표 (지역별 + 합계)</h2>
<table>
<tr><th>지역</th><th>표본수</th><th>이상치 개수(비율)</th><th>Top3 영향 요인</th></tr>
{% for r in rows %}
<tr class="{{ 'sum-row' if r.region == '합계' else '' }}">
<td>{{r.region}}</td><td>{{r.n}}</td><td>{{r.k}}</td><td>{{r.top3}}</td>
</tr>
{% endfor %}
</table>
{% endif %}

{% if region_imgs|length > 0 %}
<h2>3️⃣ 지역별 그래프</h2>
{% for item in region_imgs %}
<h3>◾{{item.region}}</h3>
<img src="{{item.path}}" alt="{{item.region}}">
{% endfor %}
{% endif %}
</body></html>
""")

rows = []
if not df_summary.empty:
    for _, rr in df_summary.iterrows():
        rows.append(dict(
            region=str(rr["지역"]),
            n=str(rr["표본수"]),
            k=str(rr["이상치 개수(비율)"]),
            top3=str(rr["Top3 영향 요인"])
        ))

region_imgs = [{"region": r, "path": region_plot_paths[r]} for r in sorted(region_plot_paths.keys())]

html = tpl.render(
    overall_img=overall_png,
    rows=rows,
    region_imgs=region_imgs
)
with open(HTML_PATH, "w", encoding="utf-8") as f:
    f.write(html)

try: webbrowser.open("file://"+HTML_PATH)
except Exception: pass

print(f"✅ HTML 저장: {HTML_PATH}")
print(f"📁 그래프 폴더: {PLOTS_DIR}")
